In [ ]:
"""
Fire Weather Hazard Detection Core Functions

Detects fire weather hazard events using FWI exceedance of day-of-year 95th percentile thresholds.
Caller handles data loading, spatial chunking, and I/O operations.
"""

import numpy as np
import pandas as pd
from typing import Tuple, Union


def detect_fwi_hazard(
    fwi_values: np.ndarray,
    thresholds: np.ndarray,
    hazard_code: int = 3,
    non_hazard_code: int = 0
) -> np.ndarray:
    """
    Detect fire weather hazard events from FWI timeseries using threshold exceedance.
    
    Flags days where FWI values meet or exceed their climatological 95th percentile threshold.
    
    Parameters
    ----------
    fwi_values : np.ndarray
        1D array of daily FWI values
    thresholds : np.ndarray
        1D array of daily 95th percentile thresholds (same length as fwi_values)
    hazard_code : int
        Value to assign for hazard days (default: 3)
    non_hazard_code : int
        Value to assign for non-hazard days (default: 0)
    
    Returns
    -------
    np.ndarray
        Integer array with hazard classification:
          - hazard_code where FWI ≥ threshold
          - non_hazard_code otherwise (including NaN positions)
    """
    # Handle NaNs: missing values cannot exceed thresholds → non-hazard
    valid_mask = ~np.isnan(fwi_values) & ~np.isnan(thresholds)
    exceedance = np.zeros(len(fwi_values), dtype=bool)
    exceedance[valid_mask] = fwi_values[valid_mask] >= thresholds[valid_mask]
    
    return np.where(exceedance, hazard_code, non_hazard_code).astype(int)


def process_grid_cell_fire_hazard(
    fwi_ts: np.ndarray,
    p95_doy: np.ndarray,
    dates: pd.DatetimeIndex,
    hazard_code: int = 3,
    non_hazard_code: int = 0
) -> np.ndarray:
    """
    Detect fire weather hazards for a single grid cell using day-of-year percentile thresholds.
    
    Aligns daily 95th percentile thresholds to dates via day-of-year mapping before detection.
    
    Parameters
    ----------
    fwi_ts : np.ndarray
        1D array of daily FWI values (time dimension only)
    p95_doy : np.ndarray
        366-element array of 95th percentile thresholds indexed by day-of-year
        (position 0 = Jan 1, position 365 = Dec 31)
    dates : pd.DatetimeIndex
        Dates corresponding to each FWI value
    hazard_code : int
        Value to assign for hazard days (default: 3)
    non_hazard_code : int
        Value to assign for non-hazard days (default: 0)
    
    Returns
    -------
    np.ndarray
        Hazard classification array aligned to input dates
    """
    # Map day-of-year to thresholds (handle leap years: Dec 31 = index 365 for both)
    day_of_year = dates.dayofyear.values - 1  # Convert to 0-based index
    day_of_year[day_of_year == 365] = 364      # Map leap day Dec 31 to non-leap index
    
    # Clip to valid range for non-leap years (365 days)
    day_of_year = np.clip(day_of_year, 0, 364)
    
    # Align thresholds to dates
    thresholds = p95_doy[day_of_year]
    
    # Detect hazards
    return detect_fwi_hazard(
        fwi_values=fwi_ts,
        thresholds=thresholds,
        hazard_code=hazard_code,
        non_hazard_code=non_hazard_code
    )